# Create AACR Awards (American Association for Cancer Research)

Creates American Association for Cancer Research grants from the AACR's own
**Funded Research** pages
(https://www.aacr.org/professionals/research-funding/funded-research/): one
WordPress page per grant program under Career Development Awards, Independent
Research Grants and Research Training and Fellowships. Each page lists grantees
under year headings ("2025 Grantees") as cards: name + degrees, position,
institution, city/country, project title, and a "Research" summary.

AACR prunes older cohorts and retires programs on the live site, so the script
also reads the Internet Archive's copies of the same program URLs (CDX-
enumerated, one snapshot per URL per year, 2021-2026). Those are AACR's own
pages as published. Live pages win over archived copies for the same grantee.

**1993-2018** come from AACR's consolidated PDF "Recipients of AACR Research
Funding & Fellowships 1993-2018" (linked from the pre-2020 aacr.org Funding
pages as "the list of all previous and current AACR grant recipients"; read
from the Internet Archive's copy, two-column layout parsed by font: bold =
year / program / grantee, regular = institution, italic = project). Team
grants list several PIs (lead + co-lead + investigators). Grants footnoted
"*Awarded by Pancreatic Cancer Action Network, administered by AACR" are
PanCAN money: kept in the parquet with `excluded = '1'`, not shipped. The
PDF's closing honors (AACR-NFCR Professorship, Landon Prizes) are not parsed.

**959 grants in the parquet, 955 shipped, 1993-2026** (local run 2026-10-01):
605 from the 1993-2018 PDF, 354 from the program pages (2015-2026; 199 of them
only in archived copies). Title 97%, institution 100%, start year 100%.
Known gaps: the PDF's last edition is 1993-2018, so 2019-2020 cohorts come only
from whatever the archived program pages still listed; the team-grant pages
(AACR-Johnson & Johnson Lung Cancer Innovation Science Grants) use a different
layout and are not parsed; 3 web cards had no year heading and were dropped.

**Partners:** most programs are co-funded (AACR-AstraZeneca, BCRF-AACR,
Lustgarten Foundation-AACR, Victoria's Secret Global Fund ... in Partnership
with Pelotonia & AACR, ...). These are AACR grants and stay under AACR; the
partner is recorded in `funder_scheme` (the program name, or the specific
variant subtitle when the page gives one).

**Excluded (not AACR money):** Stand Up To Cancer grants (Dream Teams, Catalyst,
Innovative Research Grants, Phillip A. Sharp Awards) and the Pancreatic Cancer
Collective. AACR is SU2C's scientific partner and administers them, but they are
funded by SU2C (OpenAlex F4320332898) / the Lustgarten Foundation, so they are
not attributed to AACR (§2.3.2).

**Prerequisites:**
- Run `scripts/local/aacr_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/aacr/aacr_projects.parquet` (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** citing works mostly use AACR's internal grant
number (`17-40-48-NORT` = year-mechanism-sponsor-surname; ~55% of the
crossref_work.grants / crossref_work_funders / europepmc_work_funders shells
for F4320306091). AACR does not publish that number on any grantee page, and it
cannot be derived, so a stable synthetic key is shipped:
`AACR-{year}-{person-slug}` (program slug appended only for a person with two
grants in one year). It collapses onto 0 existing shells (overlap cell below).

**No amounts:** grantee pages carry no amount (§6.7 waived; currency NULL).
`start_date` = Jan 1 of the cohort year (year-only pattern, §2.3); no end date.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320306091`
- display_name / ror_id / doi: from `openalex.funders.funders` (ror 02fnpv153, doi 10.13039/100000043)

**Priority:** `560` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.aacr_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/aacr/aacr_projects.parquet`;

In [ ]:
%sql
SELECT source, COUNT(*) as total_grants, COUNT(DISTINCT funder_award_id) as distinct_ids,
       COUNT(excluded) as excluded_not_aacr_money
FROM openalex.awards.aacr_raw GROUP BY source;

In [ ]:
%sql
DESCRIBE openalex.awards.aacr_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.aacr_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320306091 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320306091;

## Step 2: Create AACR Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.aacr_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320306091  -- American Association for Cancer Research
),

parsed AS (
    SELECT
        g.*,
        TRY_CAST(g.year AS INT) AS yr,
        -- every grantee on the card / PDF entry (team grants list 2-5 PIs).
        -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
        TRANSFORM(
            FILTER(from_json(g.people, 'array<struct<name:string,given_name:string,family_name:string,institution:string,country:string>>'),
                   p -> NULLIF(TRIM(p.family_name), '') IS NOT NULL),
            p -> named_struct(
                'given_name', NULLIF(TRIM(p.given_name), ''),
                'family_name', NULLIF(TRIM(p.family_name), ''),
                'orcid', CAST(NULL AS STRING),
                'role_start', CAST(NULL AS DATE),
                'affiliation', named_struct(
                    'name', NULLIF(TRIM(p.institution), ''),
                    'country', NULLIF(TRIM(p.country), ''),
                    'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
                )
            )
        ) AS investigators
    FROM openalex.awards.aacr_raw g
    WHERE g.excluded IS NULL  -- "*Awarded by Pancreatic Cancer Action Network, administered by AACR"
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    -- project title when the card gives one, else "<program> <year>: <grantee>" (§2.3.1)
    COALESCE(
        NULLIF(TRIM(g.project_title), ''),
        CONCAT_WS(' ', NULLIF(TRIM(COALESCE(g.program_variant, g.program)), ''), g.year) || ': ' ||
            CONCAT_WS(' ', NULLIF(TRIM(g.given_name), ''), NULLIF(TRIM(g.family_name), ''))
    ) as display_name,
    NULLIF(TRIM(g.research_summary), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE
        WHEN g.category IN ('research-training-and-fellowships', 'Research Fellowships')
             OR LOWER(g.program) LIKE '%fellow%' THEN 'fellowship'
        ELSE 'research'
    END as funding_type,
    NULLIF(TRIM(COALESCE(g.program_variant, g.program)), '') as funder_scheme,
    'aacr_funded_research' as provenance,
    CASE WHEN g.yr IS NOT NULL THEN TRY_TO_DATE(CONCAT(g.yr, '-01-01'), 'yyyy-MM-dd') END as start_date,
    CAST(NULL AS DATE) as end_date,
    g.yr as start_year,
    CAST(NULL AS INT) as end_year,
    try_element_at(g.investigators, 1) as lead_investigator,   -- safe on empty arrays (§2.3)
    try_element_at(g.investigators, 2) as co_lead_investigator,
    CASE WHEN size(g.investigators) > 0 THEN g.investigators END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM parsed g
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

In [ ]:
%sql
-- Shape check (§2.1.1): every shipped id is the synthetic AACR-{year}-{person} key; expect 0 rows.
-- Also expect 0 SU2C / Pancreatic Cancer Collective rows (excluded, not AACR money).
SELECT funder_award_id, funder_scheme, landing_page_url
FROM openalex.awards.aacr_awards
WHERE NOT funder_award_id RLIKE '^AACR-(19|20)[0-9]{2}-[a-z0-9-]+$'
   OR LOWER(COALESCE(funder_scheme, '')) RLIKE 'su2c|stand up to cancer|pancreatic cancer collective'
   OR LOWER(landing_page_url) RLIKE 'su2c|stand-up-to-cancer|pancreatic-cancer-collective';

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'aacr_funded_research' AND priority = 560;

-- Insert into openalex_awards_raw with priority.
-- Priority 560: direct-from-funder ingest of AACR's own Funded Research pages.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    560 as priority
FROM openalex.awards.aacr_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.aacr_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name, lead_investigator.affiliation.country
FROM openalex.awards.aacr_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, MIN(start_year) as first_year, MAX(start_year) as last_year
FROM openalex.awards.aacr_awards
GROUP BY funder_scheme ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.aacr_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.aacr_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.aacr_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 coverage (amount waived: not published per grant).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution,
    COUNT(lead_investigator.affiliation.country) as has_country
FROM openalex.awards.aacr_awards;

In [ ]:
%sql
-- Overlap with existing acknowledgement shells (synthetic key: expect 0 matched).
SELECT c.provenance, COUNT(*) as shells, COUNT(t.id) as matched
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.aacr_awards t ON t.id = c.id
WHERE c.funder_id = 4320306091 AND c.priority = 0
GROUP BY c.provenance;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'aacr_funded_research'
GROUP BY provenance, priority;